# Learned Fusion Ranker -- Trained From Scratch, Not Just Combining Pretrained Outputs
## Cross-validated logistic regression + gradient-boosted trees over multi-channel scores

**Why this is different from notebooks 23-29:** every hybrid approach so far (RRF fusion, full-pool rerank, the reranker ablation) combines FROZEN, pretrained components -- union candidate lists from independently pretrained embedding models, then score with an independently pretrained cross-encoder reranker. Nothing in that pipeline is actually *fit* to this specific task's queries and labels.

**This notebook trains something from scratch on this data:** treats fusion as a supervised learning-to-rank problem. Features are the per-candidate scores and ranks from each retrieval channel (MiniLM, Linq-Mistral, GTE-large, BM25) plus the existing pretrained reranker's score as one input among several -- then a model (logistic regression, and separately a gradient-boosted tree) is TRAINED on this thesis's own 101 queries to learn how to weight and combine those signals, via 5-fold GroupKFold cross-validation (grouped by query_id, so a query's candidates never appear in both train and test -- avoids any leakage/overfitting from the small query set).

**All input data is already cached** -- no new GPU compute, this runs in seconds on CPU.

In [ ]:
import json, os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

RESULT_DIR = 'result/30_learned_fusion_ranker'
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ -- ready')

## 1. Load cached channel results and build the per-query candidate pool + feature table

In [ ]:
print('[Load] Reading cached channel results...')
minilm_df = pd.read_csv('result/03_baseline_minilm/minilm_results.csv')
linq_df   = pd.read_csv('result/20_baseline_linq_mistral/20_baseline_linq_mistral_results.csv')
gte_df    = pd.read_csv('result/17_baseline_gte_large/gte_large_results.csv')
bm25_df   = pd.read_csv('result/01_baseline_bm25/bm25_results.csv')
rerank_df = pd.read_csv('result/25_hybrid_triple_full_rerank/reranked_results.csv')
production_df = pd.read_excel('dataset/production_results.xlsx')

with open('dataset/goi_search_results.json') as f:
    queries_data = json.load(f)
query_ids = [item['query_id'] for item in queries_data]
print(f'[Load] {len(query_ids)} queries')

def relevant_set(qid):
    return set(production_df[(production_df['query_id'] == qid) & (production_df['rank'] <= 1000)]['domain'])

In [ ]:
print('[Build] Constructing per-query candidate pools (union of MiniLM+Linq+GTE+BM25 top-1000) and feature table...')
rows = []
for qid in query_ids:
    m = minilm_df[minilm_df['query_id'] == qid].set_index('domain')
    l = linq_df[linq_df['query_id'] == qid].set_index('domain')
    g = gte_df[gte_df['query_id'] == qid].set_index('domain')
    b = bm25_df[bm25_df['query_id'] == qid].set_index('domain')
    r = rerank_df[rerank_df['query_id'] == qid].set_index('domain')
    pool = set(m.index) | set(l.index) | set(g.index) | set(b.index)
    rel = relevant_set(qid)
    for domain in pool:
        n_present = int(domain in m.index) + int(domain in l.index) + int(domain in g.index) + int(domain in b.index)
        rows.append({
            'query_id': qid, 'domain': domain,
            'score_minilm': m['score'].get(domain, 0.0),
            'score_linq':   l['score'].get(domain, 0.0),
            'score_gte':    g['score'].get(domain, 0.0),
            'score_bm25':   b['bm25_score'].get(domain, 0.0),
            'invrank_minilm': 1.0 / m['rank'].get(domain, 5000),
            'invrank_linq':   1.0 / l['rank'].get(domain, 5000),
            'invrank_gte':    1.0 / g['rank'].get(domain, 5000),
            'invrank_bm25':   1.0 / b['rank'].get(domain, 5000),
            'reranker_score': r['reranker_score'].get(domain, np.nan),
            'n_channels': n_present,
            'label': int(domain in rel),
        })
feat_df = pd.DataFrame(rows)
feat_df['reranker_score'] = feat_df['reranker_score'].fillna(feat_df['reranker_score'].min() - 1)
print(f'[Build] Feature table: {len(feat_df):,} (query, candidate) rows, avg pool size {len(feat_df)/len(query_ids):.0f}')
print(f'[Build] Positive rate: {feat_df["label"].mean():.3f}')

## 2. Train via 5-fold GroupKFold cross-validation (grouped by query_id -- no query leaks across train/test)

In [ ]:
feature_cols = ['score_minilm','score_linq','score_gte','score_bm25',
                 'invrank_minilm','invrank_linq','invrank_gte','invrank_bm25',
                 'reranker_score','n_channels']
X = feat_df[feature_cols].values
y = feat_df['label'].values
groups = feat_df['query_id'].values

gkf = GroupKFold(n_splits=5)
feat_df['score_logreg'] = np.nan
feat_df['score_gbdt']   = np.nan

logreg_coefs = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr = y[train_idx]

    lr = LogisticRegression(max_iter=2000, class_weight='balanced')
    lr.fit(Xtr, ytr)
    feat_df.loc[feat_df.index[test_idx], 'score_logreg'] = lr.predict_proba(Xte)[:, 1]
    logreg_coefs.append(lr.coef_[0])

    gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight='balanced', random_state=fold)
    gbdt.fit(Xtr, ytr)
    feat_df.loc[feat_df.index[test_idx], 'score_gbdt'] = gbdt.predict_proba(Xte)[:, 1]
    print(f'[Train]   fold {fold+1}/5 done -- {len(test_idx):,} held-out rows scored')

print('[Train] Cross-validated feature weights (LogisticRegression, averaged across folds):')
avg_coefs = np.mean(logreg_coefs, axis=0)
for name, coef in sorted(zip(feature_cols, avg_coefs), key=lambda x: -abs(x[1])):
    print(f'    {name:<16} {coef:+.3f}')

## 3. Evaluate held-out predictions -- k=10,50,100,300,500,1000 (same protocol as every other experiment)

In [ ]:
def precision_at_k(ret, rel, k): return len(set(ret[:k]) & rel) / k if k else 0
def recall_at_k(ret, rel, k):    return len(set(ret[:k]) & rel) / len(rel) if rel else 0
def f1_at_k(ret, rel, k):
    p, r = precision_at_k(ret, rel, k), recall_at_k(ret, rel, k)
    return 2*p*r/(p+r) if (p+r) > 0 else 0
def dcg_at_k(ret, rel, k):
    return sum(1/np.log2(i+2) for i, d in enumerate(ret[:k]) if d in rel)
def ndcg_at_k(ret, rel, k):
    ideal = dcg_at_k(list(rel), rel, k)
    return dcg_at_k(ret, rel, k) / ideal if ideal else 0

K_VALUES = [10, 50, 100, 300, 500, 1000]
for model_name, score_col in [('logreg', 'score_logreg'), ('gbdt', 'score_gbdt')]:
    eval_rows = []
    for qid in query_ids:
        sub = feat_df[feat_df['query_id'] == qid].sort_values(score_col, ascending=False)
        retrieved = sub['domain'].tolist()
        rel = relevant_set(qid)
        for k in K_VALUES:
            eval_rows.append({'query_id': qid, 'k': k,
                               'precision': precision_at_k(retrieved, rel, k),
                               'recall': recall_at_k(retrieved, rel, k),
                               'f1': f1_at_k(retrieved, rel, k),
                               'ndcg': ndcg_at_k(retrieved, rel, k)})
    eval_df = pd.DataFrame(eval_rows)
    eval_df.to_csv(f'{RESULT_DIR}/evaluation_{model_name}.csv', index=False)
    print(f'\n[Eval] === {model_name.upper()} (cross-validated, held-out predictions) ===')
    for k in K_VALUES:
        s = eval_df[eval_df['k']==k]
        print(f'  k={k:<5} recall={s["recall"].mean():.4f}  precision={s["precision"].mean():.4f}  ndcg={s["ndcg"].mean():.4f}')

feat_df.to_csv(f'{RESULT_DIR}/scored_candidates.csv', index=False)
with open(f'{RESULT_DIR}/feature_weights.json', 'w') as f:
    json.dump({name: float(c) for name, c in zip(feature_cols, avg_coefs)}, f, indent=2)
print(f'\n[Done] Saved to {RESULT_DIR}/')